In [1]:
import sys
# This ensures the libraries are installed in the EXACT python environment you are currently using
!{sys.executable} -m pip install appscript psutil xlwings upstox-python-sdk pandas requests


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [5]:
import upstox_client

# REPLACE THESE with your details from Upstox Developer Console
api_key = '76ef74f9-6187-449d-9b2f-9909e0950423'
redirect_uri = 'https://127.0.0.1:5000/' # Or whatever you set in the dashboard

print("1. Click this link and log in:")
print(f"https://api.upstox.com/v2/login/authorization/dialog?client_id={api_key}&redirect_uri={redirect_uri}&response_type=code")

1. Click this link and log in:
https://api.upstox.com/v2/login/authorization/dialog?client_id=76ef74f9-6187-449d-9b2f-9909e0950423&redirect_uri=https://127.0.0.1:5000/&response_type=code


In [ ]:
import requests

auth_code = 'IV-qpL'
api_secret = 'oytwczd85q'

url = 'https://api.upstox.com/v2/login/authorization/token'
data = {
    'code': auth_code,
    'client_id': api_key,
    'client_secret': api_secret,
    'redirect_uri': redirect_uri,
    'grant_type': 'authorization_code',
}

response = requests.post(url, data=data)
res_json = response.json()

if 'access_token' in res_json:
    ACCESS_TOKEN = res_json['access_token']
    print("Success! Your Access Token is ready.")
else:
    print("Error:", res_json)

Success! Your Access Token is ready.


In [7]:
import xlwings as xw
import pandas as pd
import time
from datetime import datetime
from upstox_client import ApiClient, Configuration, MarketDataStreamerV3

# 1. Connect to Excel (Make sure Excel is open with a blank sheet!)
try:
    wb = xw.Book() 
    sheet = wb.sheets[0]
    sheet.clear()
    sheet.range('A1').value = ["Symbol", "LTP", "5m High", "5m Low", "Candle Time", "Status"]
    print("Connected to Excel successfully.")
except Exception as e:
    print(f"Could not connect to Excel: {e}")

# 2. Define Nifty 50 Symbols
# Using a static list for reliability, or you can fetch dynamically
nifty50_symbols = ["RELIANCE", "TCS", "HDFCBANK", "ICICIBANK", "BHARTIARTL", "INFY", "ITC", "SBIN", "LICI", "HINDUNILVR"] 
# (Add the rest of the 50 symbols to this list)

instrument_keys = [f"NSE_EQ|{s}" for s in nifty50_symbols]
data_store = {k: {'high': 0, 'low': 999999, 'ltp': 0, 'time': ""} for k in instrument_keys}

# 3. Data Processing Logic
def on_market_update(data):
    now = datetime.now()
    # 5-minute bucket logic (00, 05, 10...)
    candle_time = now.replace(minute=(now.minute // 5) * 5, second=0, microsecond=0).strftime('%H:%M')

    for key, values in data['feeds'].items():
        if 'ltp' not in values: continue
        ltp = values['ltp']
        
        # Reset on new 5-min candle
        if data_store[key]['time'] != candle_time:
            data_store[key] = {'high': ltp, 'low': ltp, 'ltp': ltp, 'time': candle_time}
        else:
            data_store[key]['high'] = max(data_store[key]['high'], ltp)
            data_store[key]['low'] = min(data_store[key]['low'], ltp)
            data_store[key]['ltp'] = ltp

# 4. Start Streaming
conf = Configuration()
conf.access_token = ACCESS_TOKEN
streamer = MarketDataStreamerV3(ApiClient(conf))

streamer.on_open = lambda: streamer.subscribe(instrument_keys, mode="full")
streamer.on_message = on_market_update

print("Starting live stream... Switch to Excel to see updates.")
streamer.connect()

try:
    while True:
        # Update Excel UI every 2 seconds
        output = []
        for k in instrument_keys:
            d = data_store[k]
            output.append([k.split('|')[1], d['ltp'], d['high'], d['low'], d['time'], "Live"])
        sheet.range('A2').value = output
        time.sleep(2)
except KeyboardInterrupt:
    streamer.disconnect()
    print("Stream stopped.")

Connected to Excel successfully.
Starting live stream... Switch to Excel to see updates.
Stream stopped.


In [10]:
import xlwings as xw
import pandas as pd
import time
from datetime import datetime
import sys

# Ensure dependencies are loaded
try:
    import appscript
    import psutil
except ImportError:
    print("Dependencies missing. Installing now...")
    !{sys.executable} -m pip install appscript psutil xlwings upstox-python-sdk

from upstox_client import ApiClient, Configuration, MarketDataStreamerV3

# --- 1. SETUP YOUR CREDENTIALS ---
# Make sure you have generated this in the previous step
# ACCESS_TOKEN = 'PASTE_YOUR_ACTUAL_TOKEN_HERE' 

# --- 2. THE NIFTY 50 SYMBOLS ---
nifty50_symbols = [
    "ADANIENT", "ADANIPORTS", "APOLLOHOSP", "ASIANPAINT", "AXISBANK", "BAJAJ-AUTO", 
    "BAJFINANCE", "BAJAJFINSV", "BEL", "BHARTIARTL", "CIPLA", "COALINDIA", 
    "DRREDDY", "EICHERMOT", "ETERNAL", "GRASIM", "HCLTECH", "HDFCBANK", 
    "HDFCLIFE", "HINDALCO", "HINDUNILVR", "ICICIBANK", "INDIGO", "INFY", 
    "ITC", "JIOFIN", "JSWSTEEL", "KOTAKBANK", "LT", "M&M", 
    "MARUTI", "MAXHEALTH", "NESTLEIND", "NTPC", "ONGC", "POWERGRID", 
    "RELIANCE", "SBILIFE", "SHRIRAMFIN", "SBIN", "SUNPHARMA", "TCS", 
    "TATACONSUM", "TMPV", "TATASTEEL", "TECHM", "TITAN", "TRENT", 
    "ULTRACEMCO", "WIPRO"
]

instrument_keys = [f"NSE_EQ|{s}" for s in nifty50_symbols]
data_store = {k: {'high': 0, 'low': 999999, 'ltp': 0, 'time': "00:00"} for k in instrument_keys}

# --- 3. EXCEL SETUP ---
def setup_excel():
    print("Connecting to Excel...")
    try:
        # Connect to the active workbook (Must be open!)
        wb = xw.Book() 
        sheet = wb.sheets[0]
        sheet.clear()
        headers = ["Symbol", "LTP", "5m High", "5m Low", "Candle Window", "System Status"]
        sheet.range('A1').value = headers
        sheet.range('A1:F1').api.font_object.bold.set(True)
        sheet.range('F2').value = "Waiting for Market Ticks..."
        return sheet
    except Exception as e:
        print(f"Error: Could not find an open Excel file. {e}")
        return None

sheet = setup_excel()

# --- 4. DATA LOGIC ---
def on_market_update(data):
    now = datetime.now()
    # 5-minute bucket logic
    candle_label = now.replace(minute=(now.minute // 5) * 5, second=0, microsecond=0).strftime('%H:%M')

    for key, values in data['feeds'].items():
        if 'ltp' not in values: continue
        ltp = values['ltp']
        
        # New candle reset
        if data_store[key]['time'] != candle_label:
            data_store[key] = {'high': ltp, 'low': ltp, 'ltp': ltp, 'time': candle_label}
        else:
            data_store[key]['high'] = max(data_store[key]['high'], ltp)
            data_store[key]['low'] = min(data_store[key]['low'], ltp)
            data_store[key]['ltp'] = ltp

# --- 5. STREAMER EXECUTION ---
if sheet:
    try:
        conf = Configuration()
        conf.access_token = ACCESS_TOKEN # This must be defined in your notebook
        streamer = MarketDataStreamerV3(ApiClient(conf))

        streamer.on_open = lambda: streamer.subscribe(instrument_keys, mode="full")
        streamer.on_message = on_market_update
        
        print("Opening Stream...")
        streamer.connect()

        # The Loop that keeps the notebook alive and updates Excel
        while True:
            rows = []
            current_status = "MARKET CLOSED" if datetime.now().hour < 9 or datetime.now().hour >= 16 else "LIVE"
            
            for k in instrument_keys:
                d = data_store[k]
                rows.append([
                    k.split('|')[1], 
                    d['ltp'], 
                    d['high'], 
                    d['low'], 
                    d['time'], 
                    current_status
                ])
            
            # Flush to Excel
            sheet.range('A2').value = rows
            time.sleep(2) # Refresh every 2 seconds

    except NameError:
        print("FATAL ERROR: ACCESS_TOKEN not found. Please run the authentication cell first.")
    except KeyboardInterrupt:
        streamer.disconnect()
        print("\nStream stopped by user.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

Connecting to Excel...
Opening Stream...

Stream stopped by user.


In [11]:
import upstox_client
from upstox_client import ApiClient, Configuration, MarketDataStreamerV3
import json

# --- 1. CONFIGURATION ---
# Ensure this token is freshly generated for today!
# ACCESS_TOKEN = 'PASTE_YOUR_ACTUAL_TOKEN_HERE'

# Use specific ISIN-based keys for debugging to ensure no naming errors
# NSE_EQ|INE002A01018 is Reliance, NSE_EQ|INE467B01029 is TCS
debug_keys = ["NSE_EQ|INE002A01018", "NSE_EQ|INE467B01029"]

# --- 2. DEBUG CALLBACKS ---
def on_message(data):
    """Prints the raw data whenever a price change happens."""
    print("--- NEW TICK RECEIVED ---")
    # Pretty print the data dictionary
    print(json.dumps(data, indent=2))

def on_error(err):
    print(f"CONNECTION ERROR: {err}")

def on_open():
    print("Websocket Connection Opened! Subscribing to symbols...")
    streamer.subscribe(debug_keys, mode="full")

def on_close(close_status_code, close_msg):
    print(f"Connection Closed. Code: {close_status_code}, Msg: {close_msg}")

# --- 3. START THE TESTER ---
try:
    conf = Configuration()
    conf.access_token = ACCESS_TOKEN
    api_client = ApiClient(conf)
    
    streamer = MarketDataStreamerV3(api_client)
    
    streamer.on_open = on_open
    streamer.on_message = on_message
    streamer.on_error = on_error
    streamer.on_close = on_close

    print("Attempting to connect to Upstox Market Feed...")
    streamer.connect()

except NameError:
    print("FATAL ERROR: The variable 'ACCESS_TOKEN' is not defined. Run your auth cell first.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Attempting to connect to Upstox Market Feed...


In [12]:
import os
import ssl

if not os.environ.get('PYTHONHTTPSVERIFY', '') and getattr(ssl, '_create_unverified_context', None):
    ssl._create_default_https_context = ssl._create_unverified_context
print("SSL verification bypassed for debugging.")

SSL verification bypassed for debugging.


In [13]:
import requests
import json

# Use the token you just generated
# ACCESS_TOKEN = 'YOUR_FRESH_TOKEN'

def test_token_validity(token):
    url = 'https://api.upstox.com/v2/market-quote/quotes?symbol=NSE_EQ|INE002A01018'
    headers = {
        'accept': 'application/json',
        'Api-Version': '2.0',
        'Authorization': f'Bearer {token}'
    }
    
    try:
        response = requests.get(url, headers=headers)
        result = response.json()
        
        if response.status_code == 200 and result['status'] == 'success':
            price = result['data']['NSE_EQ:RELIANCE']['last_price']
            print(f"✅ SUCCESS: Your token is working perfectly!")
            print(f"Current Reliance Price: {price}")
            return True
        else:
            print(f"❌ ERROR: Server rejected the token.")
            print(f"Status Code: {response.status_code}")
            print(f"Reason: {result.get('errors', 'Unknown Error')}")
            return False
    except Exception as e:
        print(f"❌ SYSTEM ERROR: {e}")
        return False

test_token_validity(ACCESS_TOKEN)

✅ SUCCESS: Your token is working perfectly!
Current Reliance Price: 1403.9


True

In [19]:
import xlwings as xw
import requests
import time
from datetime import datetime

# --- 1. CONFIGURATION ---
nifty50_symbols = ["RELIANCE", "TCS", "HDFCBANK", "ICICIBANK", "INFY", "BHARTIARTL", "ITC", "SBIN", "LTIM", "HINDUNILVR"]
instrument_keys = [f"NSE_EQ|{s}" for s in nifty50_symbols]
data_store = {s: {'high': 0, 'low': 999999, 'ltp': 0, 'time': ""} for s in nifty50_symbols}

# --- 2. FORCE-CONNECTION TO EXCEL ---
def get_sheet():
    try:
        # On Mac, this forces Excel to become the frontmost app
        app = xw.App(visible=True)
        if app.books.count == 0:
            wb = app.books.add()
        else:
            wb = app.books.active
        
        sheet = wb.sheets[0]
        sheet.activate() # MAC FIX: Ensure this is the active sheet
        sheet.clear()
        
        # Set Headers
        headers = ["Symbol", "LTP", "5m High", "5m Low", "Candle", "Sync"]
        sheet.range('A1').value = headers
        print("✅ Excel Connection Verified. Look at your Excel window NOW.")
        return sheet
    except Exception as e:
        print(f"❌ Excel Error: {e}")
        return None

target_sheet = get_sheet()

# --- 3. THE DATA ENGINE ---
def fetch_now():
    keys_str = ",".join(instrument_keys)
    url = f'https://api.upstox.com/v2/market-quote/quotes?instrument_key={keys_str}'
    headers = {'Authorization': f'Bearer {ACCESS_TOKEN}', 'Accept': 'application/json'}
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return response.json()['data']
        return None
    except:
        return None

# --- 4. THE LIVE LOOP ---
if target_sheet:
    print("🚀 Starting Data Push...")
    try:
        while True:
            data = fetch_now()
            if data:
                now = datetime.now()
                candle = now.replace(minute=(now.minute // 5) * 5, second=0).strftime('%H:%M')
                
                final_rows = []
                for key in data.keys():
                    sym = key.split(':')[-1]
                    price = data[key]['last_price']
                    
                    # 5-min Logic
                    if data_store[sym]['time'] != candle:
                        data_store[sym] = {'high': price, 'low': price, 'ltp': price, 'time': candle}
                    else:
                        data_store[sym]['high'] = max(data_store[sym]['high'], price)
                        data_store[sym]['low'] = min(data_store[sym]['low'], price)
                        data_store[sym]['ltp'] = price
                    
                    final_rows.append([sym, price, data_store[sym]['high'], data_store[sym]['low'], candle, now.strftime('%H:%M:%S')])
                
                # MAC FIX: Use 'value' on a specific range size to force a refresh
                target_sheet.range('A2').value = final_rows
            
            time.sleep(1) # Faster refresh to verify connection
    except KeyboardInterrupt:
        print("Stopped.")

❌ Excel Error: Could not activate App! Try to instantiate the App with visible=True.
